In [1]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

In [2]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [3]:
from google.colab import userdata
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

In [4]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/")

In [5]:
df = pd.read_csv("/content/yt_preprocessed_data.csv").dropna(subset=['Comment'])
df.shape

(17680, 3)

In [6]:
df.drop(columns=[ "Unnamed: 0"], inplace=True)

In [7]:
df.head()

,Comment,Sentiment
0,let not forget apple pay 2014 required brand n...,neutral
1,nz 50 retailer dont even contactless credit ca...,negative
2,forever acknowledge channel help lesson idea e...,positive
3,whenever go place doesnt take apple pay doesnt...,negative
4,apple pay convenient secure easy use used kore...,positive


In [8]:
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

2026/07/12 17:38:08 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling Imbalanced Data' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/67bc891929e6418ca4896e6a95ac43cb', creation_time=1783877888790, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783877888790, lifecycle_stage='active', name='Exp 4 - Handling Imbalanced Data', tags={}, trace_location=None, workspace='default'>

In [9]:
from imblearn.over_sampling import (
    SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE, KMeansSMOTE, RandomOverSampler
)
from imblearn.under_sampling import (
    RandomUnderSampler, NearMiss, TomekLinks, EditedNearestNeighbours,
    AllKNN, InstanceHardnessThreshold
)
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier, RUSBoostClassifier

In [10]:
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3)
    max_features = 10000

    X_train, X_test, y_train, y_test = train_test_split(
        df['Comment'], df['Sentiment'], test_size=0.2, random_state=42, stratify=df['Sentiment']
    )

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    class_weight = None

    if imbalance_method == 'class_weights':
        class_weight = 'balanced'

    elif imbalance_method == 'oversampling':  # SMOTE
        X_train_vec, y_train = SMOTE(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'adasyn':
        X_train_vec, y_train = ADASYN(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'undersampling':
        X_train_vec, y_train = RandomUnderSampler(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'smote_enn':
        X_train_vec, y_train = SMOTEENN(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'smote_tomek':
        X_train_vec, y_train = SMOTETomek(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'borderline_smote':
        X_train_vec, y_train = BorderlineSMOTE(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'svm_smote':
        X_train_vec, y_train = SVMSMOTE(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'random_oversampling':
        X_train_vec, y_train = RandomOverSampler(random_state=42).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'near_miss':
        X_train_vec, y_train = NearMiss(version=2).fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'tomek_links':
        X_train_vec, y_train = TomekLinks().fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'edited_nn':
        X_train_vec, y_train = EditedNearestNeighbours().fit_resample(X_train_vec, y_train)

    elif imbalance_method == 'all_knn':
        X_train_vec, y_train = AllKNN().fit_resample(X_train_vec, y_train)

    with mlflow.start_run() as run:
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        n_estimators = 200
        max_depth = 15
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight
        )

        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(confusion_matrix_filename)
        mlflow.log_artifact(confusion_matrix_filename)
        plt.close()

        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")


imbalance_methods = [
    'class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn',
    'smote_tomek', 'borderline_smote', 'svm_smote', 'random_oversampling',
    'near_miss', 'tomek_links', 'edited_nn', 'all_knn'
]

for method in imbalance_methods:
    run_imbalanced_experiment(method)

2026/07/12 17:40:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3/runs/377ee4b42cf44f9f89eae0fedf2ebac5
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3


2026/07/12 17:42:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3/runs/dfd28a6f3e4246e4b6a2d92fcfd850f5
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3


2026/07/12 17:43:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/07/12 18:01:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_all_knn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3/runs/cc0591b97752496f8238bec3bcc99e15
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3


2026/07/12 18:03:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_balanced_rf_RandomForest_TFIDF_Trigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3/runs/f0317f79ad91468395ea854891d66dd4
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/3


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['imblearn.ensemble._forest.BalancedRandomForestClassifier', 'imblearn.pipeline.Pipeline', 'imblearn.under_sampling._prototype_selection._random_under_sampler.RandomUnderSampler'].

In [11]:
import mlflow
import pandas as pd



client = mlflow.tracking.MlflowClient()

# experiment id 3, based on your run URLs
experiment_id = "3"
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.accuracy DESC"]
)

rows = []
for run in runs:
    m = run.data.metrics
    p = run.data.params
    rows.append({
        "imbalance_method": p.get("imbalance_method"),
        "accuracy": m.get("accuracy"),
        "macro_avg_f1": m.get("macro avg_f1-score"),
        "macro_avg_precision": m.get("macro avg_precision"),
        "macro_avg_recall": m.get("macro avg_recall"),
        "neutral_f1": m.get("neutral_f1-score"),
        "neutral_precision": m.get("neutral_precision"),
        "neutral_recall": m.get("neutral_recall"),
    })

df_results = pd.DataFrame(rows).dropna(subset=["imbalance_method"])

# Combined score: weight accuracy and macro F1 equally
# macro F1 matters more than accuracy for imbalanced classes
df_results["combined_score"] = (
    0.4 * df_results["accuracy"] + 0.6 * df_results["macro_avg_f1"]
)

df_results = df_results.sort_values("combined_score", ascending=False)
print(df_results.to_string(index=False))

   imbalance_method  accuracy  macro_avg_f1  macro_avg_precision  macro_avg_recall  neutral_f1  neutral_precision  neutral_recall  combined_score
        smote_tomek  0.697115      0.606935             0.599508          0.616782    0.541057           0.527412        0.555427        0.643007
   borderline_smote  0.696267      0.606203             0.599087          0.615519    0.551916           0.531551        0.573903        0.642228
       oversampling  0.694570      0.605200             0.596968          0.616857    0.552189           0.537118        0.568129        0.640948
      class_weights  0.681844      0.610736             0.599615          0.637817    0.553862           0.494555        0.629330        0.639179
random_oversampling  0.675057      0.609319             0.597991          0.647479    0.555441           0.500000        0.624711        0.635614
        balanced_rf  0.666855      0.604637             0.595156          0.648952    0.562500           0.505525        0.6